# Ethan's JDSearch pipeline walkthrough

This notebook is for teammates reviewing the corrected work on **ethan**.
It explains the inputs, shows the actual implementation, presents saved
experiment results, and provides cells to rerun the pipeline.

**Read first:** the results below come from the previously verified
20,000-user Windows CPU run. They are not outputs of a new Colab training run.
The notebook's review cells have been checked locally; a fresh Colab/GPU
experiment has not been executed for this notebook.

You can read all explanations, source snapshots and result tables directly
in GitHub or Colab without running anything. The executable review cells
do not train models unless you enable the experiment flag.

1. Understand the task and field roles.
2. Read the timeline and complete-history construction.
3. Inspect features and evaluation before the models.
4. Compare baselines, neural controls and diagnostics.
5. Discuss what the evidence supports.

Current outcome labels are targets. They must never enter input features.
This is reranking of provided slates, not catalogue-wide retrieval.


## Setup for review

In Colab this cell clones the repository into a dedicated folder and checks
out the pinned source revision. Locally it finds the existing checkout and
does not change its branch. Pinning separates reproducible source from
future branch edits. This notebook itself can live on a newer commit.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys
from IPython.display import Markdown, display

REPO_URL = 'https://github.com/steffanylars/nus-web-mining-business-insights.git'
SOURCE_REF = '4739ff8fe2f59a4141310d60fc36bd207517b0c4'
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = Path('/content/ethan-jdsearch-review')
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', SOURCE_REF], check=True)
else:
    candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'final-report-review']
    REPO_DIR = next((p for p in candidates
        if (p / '.git').exists() and (p / 'Final Project/src').is_dir()), None)
    if REPO_DIR is None:
        raise RuntimeError('Open this notebook from the repository checkout.')

PROJECT = REPO_DIR / 'Final Project'
SAVED = PROJECT / 'artifacts/corrected'
print('Source revision:', SOURCE_REF)
print('Repository:', REPO_DIR)
print('Notebook Python:', sys.version.split()[0])


## Saved results for the team to inspect

These are the checked-in measured results of the fixed 20,000-user sample
(14,000 train / 3,000 validation / 3,000 test, seed 42). The complete test
slates contain 266,640 candidates. Ranking averages use 2,965 queries with
a positive outcome and exclude the other 35 all-negative queries.

The simple MLP control contains the **same engineered history features**;
it removes the additional neural history towers. The matched experiment
does not establish that dual attention improves on those features.


# Corrected measured results

| Model | NDCG@10 | MRR@200 | P@1 | HR@10 |
|---|---|---|---|---|
| Random | 0.0829 | 0.1034 | 0.0314 | 0.2277 |
| Popularity | 0.3045 | 0.3113 | 0.1865 | 0.5771 |
| Content/Relevance-only | 0.0951 | 0.1088 | 0.0354 | 0.2442 |
| SVD(train-only fold-in) | 0.2759 | 0.2819 | 0.1592 | 0.5356 |
| LambdaMART(features) | 0.3630 | 0.3669 | 0.2297 | 0.6516 |
| MLP(no towers) | 0.3725 | 0.3840 | 0.2519 | 0.6580 |
| Pooled history | 0.3738 | 0.3844 | 0.2513 | 0.6631 |
| Short only | 0.3713 | 0.3826 | 0.2503 | 0.6583 |
| Order only | 0.3731 | 0.3825 | 0.2496 | 0.6621 |
| DHAR(ours) | 0.3713 | 0.3831 | 0.2499 | 0.6583 |
| History-conditioned CF | 0.2694 | 0.2711 | 0.1403 | 0.5474 |

| Architecture | Validation NDCG@10 | Test NDCG@10 |
|---|---|---|
| MLP(no towers) | 0.3676 | 0.3725 |
| Pooled history | 0.3679 | 0.3738 |
| Short only | 0.3692 | 0.3713 |
| Order only | 0.3678 | 0.3731 |
| DHAR(ours) | 0.3689 | 0.3713 |

DHAR differs from the same-feature MLP by -0.0012 NDCG@10 (paired query bootstrap 95% interval [-0.0034, +0.0011]). This run does not establish an advantage for dual attention beyond the engineered features.

Sources: results_baselines_lgb.json, results_neural.json, ablation.json and paired_bootstrap.json. See REPORT.md for protocol and limitations.


In [ ]:
# Review the source-of-truth table without fitting any model.
display(Markdown((SAVED / 'RESULTS.md').read_text(encoding='utf-8')))


## Optional environment and data setup

The review cells need no raw data. To run experiments, obtain the two raw
files from the [official JDSearch source](https://github.com/rucliujn/JDsearch)
and set DATA_DIR. Google Drive can hold the files; mounting it is optional.
Raw data and large model state are not included in this notebook/repository.

INSTALL_ENV creates an isolated Python environment for the scripts, keeping
the notebook kernel's preinstalled packages separate. The requirements were
validated with Python 3.12 on Windows; Linux wheel/GPU compatibility has not
been independently verified here. Inspect installation errors and the selected
runtime rather than assuming a Colab run reproduces the saved CPU results.

Start with 1,000 users for a trial. Set N_USERS to 20,000 to match the saved
sample. The full feature build has substantial RAM requirements; a smaller
sample or a runtime with more RAM may be necessary. Smaller-run scores must
not be presented as the saved 20,000-user experiment.


In [ ]:
INSTALL_ENV = False
MOUNT_DRIVE = False
RUN_EXPERIMENTS = False
RUN_REGRESSION_CHECKS = False
N_USERS = 1000  # Set 20000 to reproduce the saved sample size.
SEED = 42

if MOUNT_DRIVE:
    if not IN_COLAB:
        raise RuntimeError('Google Drive mounting is only available in Colab.')
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = (Path('/content/drive/MyDrive/JDsearch') if IN_COLAB
            else PROJECT / 'data')
RUN_DIR = ((Path('/content/jdsearch-runs') if IN_COLAB else REPO_DIR.parent / 'jdsearch-runs')
           / f'ethan-users{N_USERS}-seed{SEED}')
EXPERIMENT_PYTHON = sys.executable

if INSTALL_ENV:
    venv_dir = (Path('/content/jdsearch-venv') if IN_COLAB
                else REPO_DIR.parent / '.venv-jdsearch-notebook')
    subprocess.run([sys.executable, '-m', 'venv', str(venv_dir)], check=True)
    EXPERIMENT_PYTHON = str(venv_dir / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python'))
    subprocess.run([EXPERIMENT_PYTHON, '-m', 'pip', 'install',
                    '-r', str(PROJECT / 'requirements.txt')], check=True)

experiment_env = os.environ.copy()
experiment_env.update(JD_USER_TXT=str(DATA_DIR / 'user_behavior_data.txt'),
                      JD_PRODUCT_TXT=str(DATA_DIR / 'product_meta_data.txt'),
                      JD_RUN_DIR=str(RUN_DIR), JD_SEED=str(SEED))
print('Raw data directory:', DATA_DIR)
print('New output directory:', RUN_DIR)
print('Training enabled:', RUN_EXPERIMENTS)
# The new run directory is separate from the published corrected evidence.


In [ ]:
def run_stage(*stages):
    if not RUN_EXPERIMENTS:
        print('Skipped experiment stages:', ', '.join(stages))
        return
    missing = [name for name in ['user_behavior_data.txt', 'product_meta_data.txt']
               if not (DATA_DIR / name).is_file()]
    if missing:
        raise FileNotFoundError(f'Place the raw files in {DATA_DIR}: {missing}')
    subprocess.run([EXPERIMENT_PYTHON, '-u', str(PROJECT / 'src/run_pipeline.py'),
                    '--users', str(N_USERS), '--stages', *stages],
                   cwd=REPO_DIR, env=experiment_env, check=True)


## 1 What the historical fields mean

Each candidate pair has a current query, product ID and outcome label.
A user's preceding events have item IDs, action types, optional historical
queries and consecutive time gaps. Term IDs inside text fields are separated
by **U+0018**; underscore separates list elements, not terms.

If n events have gaps [0, g1, ..., gn], the final gap ends at the current
query. The age of event i is the sum of all subsequent gaps. For [0,12,4,5],
event ages are [21,9,5] seconds. Using a single gap as an event age is wrong.

The shared short channel is CLICK <=7 days or CART <=90 days. The order
channel uses preceding ORD events of any observed age. Sequence sampling
happens separately within each complete horizon.


### Source to read: data_utils.py

[Canonical source at the pinned revision](https://github.com/steffanylars/nus-web-mining-business-insights/blob/4739ff8fe2f59a4141310d60fc36bd207517b0c4/Final%20Project/src/data_utils.py). The following is a readable source snapshot, not an executed training cell. The run cell below calls this script from the checked-out repository.

```python
"""JDSearch field semantics shared by training and inference."""
import numpy as np
import pandas as pd

def split_list(value):
    return str(value).split('_') if value is not None and str(value).strip() else []

def tok(value):
    if value is None or pd.isna(value) or str(value).strip() in {'', '-1'}:
        return set()
    return {part for part in str(value).split('\x18') if part}

def elapsed_days(gaps, n_events):
    values = np.asarray(gaps, dtype=np.float64)
    if len(values) != n_events + 1 or not np.isfinite(values).all() or (values < 0).any() or values[0] != 0:
        raise ValueError('History requires n+1 finite nonnegative gaps with initial zero')
    return np.cumsum(values[::-1])[::-1][1:] / 86400.0

def short_mask(history):
    """Intent channel: clicks <=7 days or carts <=90 days."""
    return ((history['type'].eq('CLICK') & history.elapsed_days.le(7)) |
            (history['type'].eq('CART') & history.elapsed_days.le(90)))

def select_sequence(values, length):
    """Evenly sample the complete chronological horizon, keeping both endpoints."""
    if length < 1:
        raise ValueError('Sequence length must be positive')
    values = np.asarray(values)
    if len(values) <= length:
        return values
    return values[np.linspace(0, len(values) - 1, length, dtype=int)]
```


In [ ]:
# A small illustrative calculation, not a new dataset/model result.
from itertools import accumulate
query = '12\x1832\x1856'
gaps = [0, 12, 4, 5]
ages_seconds = list(reversed(list(accumulate(reversed(gaps)))))[1:]
assert query.split('\x18') == ['12', '32', '56']
assert ages_seconds == [21, 9, 5]
print('Query terms:', query.split('\x18'))
print('Event ages before query:', ages_seconds, 'seconds')


## 2 Temporal analysis and the instructor's feedback

The full-data analysis finds sharp changes near seven days for clicks and
90 days for carts. Rare events remain outside those approximate windows.
This is consistent with differing logging/retention windows; it does not
verify JD's actual storage policy. Missing older clicks are unobserved,
not negative preference evidence. Historical input semantics are separate
from the current outcome gains used by the ranker.

Stage 1 gives broad age bins; stage 1b gives daily counts. Both scan the
full raw behavior file even when later modeling uses a smaller sample.


### Source to read: step1_preprocess_temporal.py

[Canonical source at the pinned revision](https://github.com/steffanylars/nus-web-mining-business-insights/blob/4739ff8fe2f59a4141310d60fc36bd207517b0c4/Final%20Project/src/step1_preprocess_temporal.py). The following is a readable source snapshot, not an executed training cell. The run cell below calls this script from the checked-out repository.

```python
"""
BT4222 Final Project - Group 13 - JDSearch
Step 1: Parsing + Temporal History Analysis (addresses the instructor's feedback)

The instructor noted that in JDSearch, historical ORDERS and CARTS dramatically
outnumber CLICKS, with millions of orders/carts having no recorded clicks.
They asked us to trace the history timeline back in time: convert the consecutive
intervals (seconds in `history_time_list`) into elapsed days BEFORE the final
test query, and watch what happens to the click / cart / order distribution as
we trace user activity further back.

Semantics (verified on the raw file):
  history_{qry,wid,type}_list each have n values (one per historical event).
  history_time_list has n+1 values: [0, gap(e0->e1), ..., gap(e_{n-1}->test_query)].
  => elapsed_seconds_before_query[i] = sum(time[i+1:])   (suffix sum)
     elapsed_days[i] = elapsed_seconds[i] / 86400.

Outputs (written to ../artifacts and ../figures):
  - temporal_type_by_daybin.csv       : composition of CLICK/CART/ORD/FLW per elapsed-day bin
  - temporal_summary.txt               : headline numbers for the report
  - fig_temporal_composition.png       : stacked composition vs elapsed days
  - fig_temporal_clickshare.png        : click share decay vs elapsed days
"""
import os, sys, time, json
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from config import SRC_USER, RES as ART, FIG, prepare
from data_utils import elapsed_days

TYPES = ["CLICK", "CART", "ORD", "FLW"]
# elapsed-day bin edges
EDGES = [0, 1/1440, 1/24, 1, 7, 30, 90, 180, 365, np.inf]  # min, hour, day, week, month...
LABELS = ["<1min", "1min-1h", "1h-1d", "1-7d", "7-30d", "30-90d", "90-180d", "180-365d", ">365d"]

def main():
    prepare()
    t0 = time.time()
    print("Loading user_behavior_data.txt (919MB)...", flush=True)
    df = pd.read_csv(SRC_USER, sep="\t", dtype=str, low_memory=False)
    n_users = len(df)
    print(f"  users: {n_users:,}  ({time.time()-t0:.0f}s)", flush=True)

    # global accumulators
    # counts[type][bin_index] = number of events
    counts = np.zeros((len(TYPES), len(LABELS)), dtype=np.int64)
    type_idx = {t: i for i, t in enumerate(TYPES)}
    total_events = 0
    total_time_zero = 0
    # also: per-event elapsed_days reservoir for a continuous plot (sample)
    sample_elapsed = []   # (elapsed_days, type_idx)
    rng = np.random.default_rng(42)

    # headline: among events in the OLDEST tercile of each history, click share
    old_counts = np.zeros(len(TYPES), dtype=np.int64)   # events older than 30 days
    recent_counts = np.zeros(len(TYPES), dtype=np.int64)  # events within 1 day

    def split(v):
        if v is None or (isinstance(v, float)) or str(v) == "nan" or str(v).strip() == "":
            return []
        return str(v).split("_")

    t1 = time.time()
    for ridx, (types_s, times_s) in enumerate(zip(df["history_type_list"].values,
                                                  df["history_time_list"].values)):
        types = split(types_s)
        times = split(times_s)
        n = len(types)
        if n == 0:
            continue
        gaps = np.asarray([float(x) for x in times])
        days_checked = elapsed_days(gaps, n)
        elapsed = days_checked * 86400.
        total_time_zero += int((gaps[1:] == 0).sum())
        days = elapsed / 86400.0
        bins = np.digitize(days, EDGES[1:], right=False)  # 0..len(LABELS)-1
        bins = np.clip(bins, 0, len(LABELS) - 1)
        for ti, b in zip(types, bins):
            ci = type_idx.get(ti)
            if ci is None:
                continue
            counts[ci, b] += 1
            total_events += 1
        # recent vs old headline
        d = days
        for ti, dd in zip(types, d):
            ci = type_idx.get(ti)
            if ci is None:
                continue
            if dd <= 1.0:
                recent_counts[ci] += 1
            elif dd > 30.0:
                old_counts[ci] += 1
        # reservoir sample for continuous plot
        if ridx % 20 == 0:
            k = min(n, 30)
            sel = rng.choice(n, size=k, replace=False)
            for s in sel:
                ci = type_idx.get(types[s])
                if ci is not None:
                    sample_elapsed.append((days[s], ci))
        if ridx % 20000 == 0 and ridx > 0:
            print(f"  processed {ridx:,}/{n_users:,} users  ({time.time()-t1:.0f}s)", flush=True)

    print(f"Parsed all histories. total events={total_events:,}  ({time.time()-t0:.0f}s)", flush=True)

    # ---- build composition table ----
    col_tot = counts.sum(axis=0)  # events per bin
    comp = counts / np.maximum(col_tot, 1)  # share of each type within bin
    tbl = pd.DataFrame(counts.T, index=LABELS, columns=TYPES)
    tbl["total_events"] = col_tot
    for t in TYPES:
        tbl[f"{t}_share"] = (counts[type_idx[t]] / np.maximum(col_tot, 1))
    tbl.to_csv(ART / "temporal_type_by_daybin.csv")
    print("\n=== Interaction-type composition by elapsed-days-before-query bin ===")
    print(tbl[[*TYPES, "total_events", "CLICK_share", "CART_share", "ORD_share"]].to_string())

    # ---- headline numbers ----
    rec_tot = recent_counts.sum(); old_tot = old_counts.sum()
    lines = []
    lines.append("TEMPORAL HISTORY ANALYSIS - headline numbers (full data, 173,831 users)")
    lines.append("")
    lines.append(f"Total historical events parsed: {total_events:,}")
    lines.append("")
    lines.append("Within <=1 day before the test query:")
    for t in TYPES:
        c = recent_counts[type_idx[t]]
        lines.append(f"   {t:6s}: {c:>12,}  ({100*c/max(rec_tot,1):5.2f}% of recent events)")
    lines.append(f"   recent click share = {100*recent_counts[type_idx['CLICK']]/max(rec_tot,1):.2f}%")
    lines.append("")
    lines.append("More than 30 days before the test query:")
    for t in TYPES:
        c = old_counts[type_idx[t]]
        lines.append(f"   {t:6s}: {c:>12,}  ({100*c/max(old_tot,1):5.2f}% of old events)")
    lines.append(f"   old click share = {100*old_counts[type_idx['CLICK']]/max(old_tot,1):.2f}%")
    lines.append("")
    lines.append("Click share by bin (near -> far in time):")
    for lab in LABELS:
        lines.append(f"   {lab:10s}: CLICK {100*tbl.loc[lab,'CLICK_share']:5.2f}%  "
                     f"CART {100*tbl.loc[lab,'CART_share']:5.2f}%  "
                     f"ORD {100*tbl.loc[lab,'ORD_share']:5.2f}%  "
                     f"(n={int(tbl.loc[lab,'total_events']):,})")
    txt = "\n".join(lines)
    (ART / "temporal_summary.txt").write_text(txt)
    print("\n" + txt)

    # ---- Figure 1: stacked composition vs bin ----
    fig, ax = plt.subplots(figsize=(9, 5))
    bottom = np.zeros(len(LABELS))
    colors = {"CLICK": "#2b7bba", "CART": "#2ca089", "ORD": "#d2691e", "FLW": "#999999"}
    for t in TYPES:
        vals = tbl[f"{t}_share"].values * 100
        ax.bar(LABELS, vals, bottom=bottom, label=t, color=colors[t])
        bottom += vals
    ax.set_ylabel("Share of events within bin (%)")
    ax.set_xlabel("Elapsed time before the test query")
    ax.set_title("Interaction-type composition as history is traced back in time")
    ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.28))
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    fig.savefig(FIG / "fig_temporal_composition.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # ---- Figure 2: click share decay line ----
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(LABELS))
    ax.plot(x, tbl["CLICK_share"].values * 100, "o-", color=colors["CLICK"], label="CLICK")
    ax.plot(x, tbl["CART_share"].values * 100, "s-", color=colors["CART"], label="CART")
    ax.plot(x, tbl["ORD_share"].values * 100, "^-", color=colors["ORD"], label="ORD")
    ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=30, ha="right")
    ax.set_ylabel("Share within bin (%)"); ax.set_xlabel("Elapsed time before the test query")
    ax.set_title("Observed action composition changes with history age")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(FIG / "fig_temporal_clickshare.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"\nDONE step1 temporal. ({time.time()-t0:.0f}s total)")

if __name__ == "__main__":
    main()
```


In [ ]:
run_stage('1', '1b')


## 3 Build the dataset and preserve complete histories

Read build_records first: it aligns IDs/types/queries/times, preserves every
history event, keeps source-row identity and splits whole users. main then
reads raw product metadata for referenced products, including category names.

The old shared latest-60-event cap removed most orders before the long-term
tower saw them. It is removed here. For the saved 20,000-user sample, history
length increases from 907,554 capped events to 3,091,153 complete events.
The sample and split membership remain the same.


### Source to read: step2_build_dataset.py

[Canonical source at the pinned revision](https://github.com/steffanylars/nus-web-mining-business-insights/blob/4739ff8fe2f59a4141310d60fc36bd207517b0c4/Final%20Project/src/step2_build_dataset.py). The following is a readable source snapshot, not an executed training cell. The run cell below calls this script from the checked-out repository.

```python
"""Build a fixed user split, complete histories and a raw-catalogue metadata subset."""
import csv
import hashlib
import sys
import time
import numpy as np
import pandas as pd
from config import OUT, RES, SRC_USER, SRC_PRODUCT, SEED, SCHEMA_VERSION, prepare, save_json
from data_utils import split_list, elapsed_days

def build_records(records, selected):
    candidates, histories, sessions = [], [], []
    selected = set(selected)
    sid = 0
    for source_row, row in enumerate(records):
        if source_row not in selected:
            continue
        cw = split_list(row['candidate_wid_list'])
        labs = [int(float(x)) for x in split_list(row['candidate_label_list'])]
        hw = split_list(row['history_wid_list'])
        ht = split_list(row['history_type_list'])
        hq = split_list(row['history_qry_list'])
        if not cw or len(cw) != len(labs) or len(set(cw)) != len(cw):
            raise ValueError(f'Invalid candidate alignment at source row {source_row}')
        if len(hw) != len(ht) or len(hw) != len(hq) or any(x not in {'CLICK','CART','ORD','FLW'} for x in ht):
            raise ValueError(f'Invalid history alignment at source row {source_row}')
        if any(x not in {0,1,2,3} for x in labs):
            raise ValueError(f'Invalid labels at source row {source_row}')
        age = elapsed_days([float(x) for x in split_list(row['history_time_list'])], len(hw))
        candidates.extend((sid,row['query'],w,lab) for w,lab in zip(cw,labs))
        histories.extend((sid,pos,w,t,q,float(d)) for pos,(w,t,q,d) in enumerate(zip(hw,ht,hq,age)))
        sessions.append((sid,source_row,row['query'],len(cw),len(hw)))
        sid += 1
    cand = pd.DataFrame(candidates, columns=['session_id','query','cand_wid','label'])
    hist = pd.DataFrame(histories, columns=['session_id','pos','wid','type','qry','elapsed_days'])
    sess = pd.DataFrame(sessions, columns=['session_id','source_row_id','query','slate_size','n_hist'])
    perm = np.random.default_rng(SEED).permutation(len(sess))
    a = int(.70*len(sess)); b = a + int(.15*len(sess))
    split = np.empty(len(sess), dtype=object)
    split[perm[:a]], split[perm[a:b]], split[perm[b:]] = 'train', 'val', 'test'
    sess['split'] = split
    return cand, hist, sess

def main():
    prepare(); start = time.time()
    n_users = int(sys.argv[1]) if len(sys.argv)>1 else 20000
    if n_users < 10:
        raise ValueError('Use at least 10 users for a three-way split')
    with SRC_USER.open('rb') as f:
        n_rows = sum(1 for _ in f)-1
    idx = np.sort(np.random.default_rng(SEED).choice(n_rows, min(n_rows,n_users), replace=False))
    with SRC_USER.open(encoding='utf-8', newline='') as f:
        cand,hist,sess = build_records(csv.DictReader(f, delimiter='\t'), idx.tolist())
    print(f'Built {len(sess):,} users; {len(hist):,} complete history events', flush=True)
    refs = set(cand.cand_wid) | set(hist.wid)
    metadata = []
    with SRC_PRODUCT.open('rb') as f:
        headers = next(f).decode('utf-8').rstrip('\r\n').split('\t')
        for line in f:
            wid = line.split(b'\t',1)[0].decode('utf-8')
            if wid in refs:
                fields = line.decode('utf-8').rstrip('\r\n').split('\t')
                if len(fields) != len(headers):
                    raise ValueError(f'Invalid metadata alignment for {wid}')
                metadata.append(fields)
    meta = pd.DataFrame(metadata, columns=headers).drop_duplicates('wid')
    if not {'cate_name_3','brand_name','name'}.issubset(meta.columns):
        raise ValueError('Raw metadata requires names as well as IDs')
    for name,frame in [('candidates',cand),('history',hist),('sessions',sess),('meta_subset',meta)]:
        frame.to_parquet(OUT/f'{name}.parquet', index=False)
    membership = sess[['session_id','source_row_id','split']].to_csv(index=False)
    save_json(RES/'dataset_manifest.json', dict(schema_version=SCHEMA_VERSION, seed=SEED,
        source_users=n_rows, sampled_users=len(sess), candidate_rows=len(cand), history_rows=len(hist),
        history_policy='all preceding events; independent horizon selection occurs in neural packing',
        split_counts=sess.groupby('split').size().to_dict(),
        split_sha256=hashlib.sha256(membership.encode()).hexdigest(),
        candidate_metadata_coverage=float(cand.cand_wid.isin(set(meta.wid)).mean()),
        raw_files={p.name: {'bytes':p.stat().st_size} for p in [SRC_USER,SRC_PRODUCT]}))
    print(f'Dataset ready in {OUT} ({time.time()-start:.0f}s)', flush=True)

if __name__ == '__main__':
    main()
```


In [ ]:
run_stage('2')


## 4 Engineer the candidate features

Read the training-only popularity block, then the per-user profiles, then
the candidate feature loop. Content features compare real term sets. Short
affinity uses the shared short mask; order affinity uses all preceding orders.
Training-only popularity is shared across users; personal profiles use each
user's own available history, including at held-out inference.

h_min_elapsed is the age of the user's last event. It is not the last time
this particular candidate was seen. Current labels are carried alongside
feature rows for training/evaluation and are excluded from predictors.


### Source to read: step3_features.py

[Canonical source at the pinned revision](https://github.com/steffanylars/nus-web-mining-business-insights/blob/4739ff8fe2f59a4141310d60fc36bd207517b0c4/Final%20Project/src/step3_features.py). The following is a readable source snapshot, not an executed training cell. The run cell below calls this script from the checked-out repository.

```python
"""Features from complete preceding histories; short clicks <=7d / carts <=90d, long orders."""
import time
import numpy as np, pandas as pd

from config import OUT, RES, prepare, save_json
from data_utils import tok, short_mask

def main():
    prepare()
    t0=time.time()
    cand = pd.read_parquet(OUT/"candidates.parquet")
    hist = pd.read_parquet(OUT/"history.parquet")
    sess = pd.read_parquet(OUT/"sessions.parquet")
    meta = pd.read_parquet(OUT/"meta_subset.parquet")
    print(f"loaded: cand {len(cand):,}, hist {len(hist):,}, sess {len(sess):,}, meta {len(meta):,} ({time.time()-t0:.0f}s)")

    meta = meta.set_index("wid")
    # ---- maps wid -> category / brand / name-tokens ----
    cate3 = meta["cate_id_3"].to_dict()
    brand = meta["brand_id"].to_dict()
    name_tok = {w: tok(n) for w,n in meta["name"].items()}
    cname3 = {w: tok(n) for w,n in meta["cate_name_3"].items()} if "cate_name_3" in meta else {}
    bname  = {w: tok(n) for w,n in meta["brand_name"].items()} if "brand_name" in meta else {}

    train_sids = set(sess.loc[sess.split=="train","session_id"])

    # ---- POPULARITY PRIORS from TRAIN histories only (no leakage) ----
    hist_tr = hist[hist.session_id.isin(train_sids)]
    wid_pop = hist_tr["wid"].value_counts().to_dict()
    hist_tr_c3 = hist_tr["wid"].map(cate3)
    cate3_pop = hist_tr_c3.value_counts().to_dict()
    hist_tr_b = hist_tr["wid"].map(brand)
    brand_pop = hist_tr_b.value_counts().to_dict()
    print(f"priors built ({time.time()-t0:.0f}s)")

    # ---- Per-session history profiles (temporal-aware) ----
    # attach category/brand to history
    hist = hist.copy()
    hist["c3"] = hist["wid"].map(cate3)
    hist["b"]  = hist["wid"].map(brand)
    hist["is_click"] = (hist["type"]=="CLICK")
    hist["is_cart"]  = (hist["type"]=="CART")
    hist["is_ord"]   = (hist["type"]=="ORD")
    hist["recent7"]  = (hist["elapsed_days"]<=7)
    hist["intent"] = short_mask(hist)

    # per-session aggregates
    g = hist.groupby("session_id")
    sess_feat = pd.DataFrame({
        "h_len": g.size(),
        "h_n_click": g["is_click"].sum(),
        "h_n_cart":  g["is_cart"].sum(),
        "h_n_ord":   g["is_ord"].sum(),
        "h_n_recent7": g["recent7"].sum(),
        "h_n_distinct_c3": g["c3"].nunique(),
        "h_min_elapsed": g["elapsed_days"].min(),
        "h_queryless": g.apply(lambda d: (d["qry"]=="-1").mean(), include_groups=False),
    }).reset_index()

    # per-session horizon category/brand sets + recency-weighted affinity dicts
    # short-term intent categories (recent clicks+carts, <=7d emphasises clicks)
    short_c3 = g.apply(lambda d: set(d.loc[d.intent,"c3"].dropna()), include_groups=False).to_dict()
    short_b  = g.apply(lambda d: set(d.loc[d.intent,"b"].dropna()), include_groups=False).to_dict()
    # long-term preference categories/brands from ORDERS (any age)
    long_c3  = g.apply(lambda d: set(d.loc[d.is_ord,"c3"].dropna()), include_groups=False).to_dict()
    long_b   = g.apply(lambda d: set(d.loc[d.is_ord,"b"].dropna()), include_groups=False).to_dict()
    # All exact item IDs from preceding history
    seen_wid = g["wid"].apply(set).to_dict()
    # recency-decayed category affinity (tau=30d) and brand affinity
    aff_c3 = {}; aff_b = {}
    for sid, d in g:
        wv = np.exp(-d["elapsed_days"].values/30.0)
        c3v = d["c3"].values; bv = d["b"].values
        ac={}; ab={}
        for c,w in zip(c3v,wv):
            if pd.notna(c): ac[c]=ac.get(c,0)+w
        for b,w in zip(bv,wv):
            if pd.notna(b): ab[b]=ab.get(b,0)+w
        aff_c3[sid]=ac; aff_b[sid]=ab
    # All preceding history-query tokens per session
    sess_qtok = g.apply(lambda d: set().union(*[tok(q) for q in d["qry"] if q!="-1"]) if (d["qry"]!="-1").any() else set(), include_groups=False).to_dict()
    print(f"session profiles built ({time.time()-t0:.0f}s)")

    # ---- build feature rows per candidate ----
    cand = cand.merge(sess[["session_id","split","slate_size"]], on="session_id", how="left")
    rows = []
    cw = cand["cand_wid"].values; qs = cand["query"].values; sids = cand["session_id"].values
    lab_values = cand["label"].values; split_values = cand["split"].values
    slate_values = cand["slate_size"].values
    query_tokens = {q: tok(q) for q in cand["query"].unique()}
    for i in range(len(cand)):
        w = cw[i]; sid = sids[i]
        qt = query_tokens[qs[i]]
        c3 = cate3.get(w); b = brand.get(w)
        has_meta = 1 if w in name_tok else 0
        # content relevance
        nt = name_tok.get(w,set()); ct = cname3.get(w,set()); bt = bname.get(w,set())
        ov_name = len(qt & nt); ov_cat = len(qt & ct); ov_brand = len(qt & bt)
        jacc_name = ov_name/max(len(qt|nt),1)
        # popularity priors (log)
        pop_w = np.log1p(wid_pop.get(w,0))
        pop_c3 = np.log1p(cate3_pop.get(c3,0))
        pop_b = np.log1p(brand_pop.get(b,0))
        # short/long affinity
        sc = short_c3.get(sid,set()); sb = short_b.get(sid,set())
        lc = long_c3.get(sid,set()); lb = long_b.get(sid,set())
        short_cat_hit = 1 if (c3 is not None and c3 in sc) else 0
        short_brand_hit = 1 if (b is not None and b in sb) else 0
        long_cat_hit  = 1 if (c3 is not None and c3 in lc) else 0
        long_brand_hit = 1 if (b is not None and b in lb) else 0
        aff_cat = aff_c3.get(sid,{}).get(c3,0.0)
        aff_brand = aff_b.get(sid,{}).get(b,0.0)
        # exact re-engagement
        seen = 1 if w in seen_wid.get(sid,set()) else 0
        # query overlap with user's past queries
        q_hist_overlap = len(qt & sess_qtok.get(sid,set()))
        rows.append((sid, w, lab_values[i], split_values[i],
                     len(qt), slate_values[i], has_meta,
                     ov_name, ov_cat, ov_brand, jacc_name,
                     pop_w, pop_c3, pop_b,
                     short_cat_hit, short_brand_hit, long_cat_hit, long_brand_hit,
                     aff_cat, aff_brand, seen, q_hist_overlap))
        if i % 200000 == 0 and i>0:
            print(f"  feat {i:,}/{len(cand):,} ({time.time()-t0:.0f}s)", flush=True)

    cols = ["session_id","cand_wid","label","split",
            "query_len","slate_size","has_meta",
            "ov_name","ov_cat","ov_brand","jacc_name",
            "pop_wid","pop_cate3","pop_brand",
            "short_cat_hit","short_brand_hit","long_cat_hit","long_brand_hit",
            "aff_cat_decay","aff_brand_decay","seen_before","q_hist_overlap"]
    feat = pd.DataFrame(rows, columns=cols)
    feat = feat.merge(sess_feat, on="session_id", how="left")
    hist_cols = ["h_len","h_n_click","h_n_cart","h_n_ord","h_n_recent7","h_n_distinct_c3","h_min_elapsed","h_queryless"]
    for c in hist_cols: feat[c] = feat[c].fillna(0)
    numeric = [c for c in feat if c not in {"session_id","cand_wid","split","label"}]
    feat[numeric] = feat[numeric].astype("float32")
    if not np.isfinite(feat[numeric].to_numpy()).all():
        raise ValueError("Nonfinite numeric features")
    feat.to_parquet(OUT/"features.parquet", index=False)
    save_json(RES/"feature_summary.json", {
        "rows": len(feat), "numeric_features": numeric,
        "means_by_label": {str(k): {c:float(v) for c,v in row.items()}
                          for k,row in feat.groupby("label")[numeric].mean().iterrows()},
        "nonzero_content_pairs": {c:int((feat[c]>0).sum()) for c in ["ov_name","ov_cat","ov_brand"]},
        "short_policy": "CLICK <=7d or CART <=90d; all preceding orders in long profile",
        "h_min_elapsed_definition": "age of most recent user event, not candidate-specific recency"
    })

    print("\n=== FEATURES SUMMARY ===")
    print(f"rows: {len(feat):,}  cols: {feat.shape[1]}")
    print(f"splits: {feat.split.value_counts().to_dict()}")
    print("mean feature values by label (sample):")
    show = ["ov_name","jacc_name","pop_wid","short_cat_hit","long_cat_hit","aff_cat_decay","seen_before","has_meta"]
    print(feat.groupby("label")[show].mean().to_string())
    print(f"DONE step3 ({time.time()-t0:.0f}s)")

if __name__ == "__main__":
    main()
```


In [ ]:
run_stage('3')


## 5 Define a fair evaluation before comparing models

NDCG@10 uses gains 2^label-1. MRR@200, P@1 and HR@10 treat label >0 as relevant.
Macro averages use positive queries only. Validation and final evaluation
use the same policy. Full supplied candidate slates remain intact at evaluation.

Original candidate order is label-associated. Equal scores must therefore
not inherit that order. tie_keys uses seeded query/candidate identities so
a row permutation cannot change the tied ordering.


### Source to read: eval_utils.py

[Canonical source at the pinned revision](https://github.com/steffanylars/nus-web-mining-business-insights/blob/4739ff8fe2f59a4141310d60fc36bd207517b0c4/Final%20Project/src/eval_utils.py). The following is a readable source snapshot, not an executed training cell. The run cell below calls this script from the checked-out repository.

```python
"""Macro query metrics with order-invariant deterministic tie breaking."""
import numpy as np
import pandas as pd

def tie_keys(df, seed=0, group_col='session_id'):
    if 'cand_wid' not in df:
        raise ValueError('Candidate IDs are required for order-independent tie breaking')
    keys = df[[group_col,'cand_wid']].astype(str).assign(_seed=str(seed))
    return pd.util.hash_pandas_object(keys, index=False).to_numpy(dtype=np.uint64)

def ranked_order(group, score_col, seed=0):
    scores = group[score_col].to_numpy(dtype=float)
    if not np.isfinite(scores).all():
        raise ValueError('Ranking scores must be finite')
    return np.lexsort((tie_keys(group,seed), -scores))

def _ranked_labels(labels, scores, tiebreak):
    return np.asarray(labels)[np.lexsort((tiebreak, -np.asarray(scores)))]

def _dcg(rels, k):
    rels = np.asarray(rels, dtype=float)[:k]
    return float(((2**rels-1) / np.log2(np.arange(2,len(rels)+2))).sum())

def _metrics_from_ranked(ranked, ks):
    ranked = np.asarray(ranked)
    ideal = np.sort(ranked)[::-1]
    result = {}
    for k in ks:
        idcg = _dcg(ideal,k)
        result[f'NDCG@{k}'] = _dcg(ranked,k)/idcg if idcg else 0.
    nz = np.flatnonzero(ranked[:200] > 0)
    result['MRR@200'] = 1./(nz[0]+1) if nz.size else 0.
    result['P@1'] = float(ranked[0] > 0) if len(ranked) else 0.
    result['HR@10'] = float((ranked[:10] > 0).any())
    return result

def per_query(df, score_col, label_col='label', group_col='session_id', ks=(10,), seed=0):
    if df.duplicated([group_col,'cand_wid']).any():
        raise ValueError('Duplicate query/candidate keys')
    records = []
    # Compute all hashes once, not once per group.
    frame = df.copy()
    frame['_tie'] = tie_keys(frame,seed,group_col)
    if not np.isfinite(frame[score_col].to_numpy(dtype=float)).all():
        raise ValueError('Ranking scores must be finite')
    for sid,g in frame.groupby(group_col,sort=True):
        labels = g[label_col].to_numpy()
        if not (labels > 0).any():
            continue
        ranked = _ranked_labels(labels,g[score_col].to_numpy(),g['_tie'].to_numpy())
        records.append({group_col:sid, **_metrics_from_ranked(ranked,ks)})
    return pd.DataFrame(records, columns=[group_col,*[f'NDCG@{k}' for k in ks],
                                         'MRR@200','P@1','HR@10'])

def evaluate(df, score_col, label_col='label', group_col='session_id', ks=(10,), seed=0):
    rows = per_query(df,score_col,label_col,group_col,ks,seed)
    cols = [c for c in rows if c!=group_col]
    return {c:float(rows[c].mean()) if len(rows) else 0. for c in cols},len(rows)

def lightgbm_metric(frame):
    """Same positive-query policy and tie keys as final evaluation, vectorized."""
    groups = list(frame.groupby('session_id',sort=False).indices.values())
    width = max(map(len,groups))
    idx = np.zeros((len(groups),width),dtype=int)
    mask = np.zeros_like(idx,dtype=bool)
    for i,ids in enumerate(groups):
        idx[i,:len(ids)] = ids; mask[i,:len(ids)] = True
    labels = frame.label.to_numpy()[idx]
    labels = np.where(mask,labels,0)
    ties = tie_keys(frame)[idx]
    gain = 2.**labels-1
    discount = 1./np.log2(np.arange(2,min(width,10)+2))
    ideal = np.sort(gain,axis=1)[:,::-1][:,:10]
    idcg = (ideal*discount).sum(1); positive = idcg>0
    def metric(prediction, dataset):
        scores = np.where(mask,np.asarray(prediction)[idx],-np.inf)
        order = np.lexsort((ties,-scores),axis=1)[:,:10]
        ranked = np.take_along_axis(gain,order,axis=1)
        value = ((ranked*discount).sum(1)[positive]/idcg[positive]).mean() if positive.any() else 0.
        return 'positive_query_ndcg10',float(value),True
    return metric
```


## 6 Baselines and LambdaMART

Compare random, training-history popularity, content overlap, training-only
SVD with held-out history fold-in, and the feature-based ranker. Shared SVD
factors are fit on training histories; a held-out user's preceding history
is transformed into that fixed space. It is not a joint test-history fit.

LambdaMART selects its iteration using the validation metric. Test scores
are for reporting, not choosing settings.


### Source to read: step4_baselines_lgbm.py

[Canonical source at the pinned revision](https://github.com/steffanylars/nus-web-mining-business-insights/blob/4739ff8fe2f59a4141310d60fc36bd207517b0c4/Final%20Project/src/step4_baselines_lgbm.py). The following is a readable source snapshot, not an executed training cell. The run cell below calls this script from the checked-out repository.

```python
"""Training-only SVD with held-out fold-in, baselines and LambdaMART."""
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
import joblib
from config import OUT, RES, SEED, THREADS, feature_columns, prepare, save_json
from eval_utils import evaluate, tie_keys, lightgbm_metric

LGB_PARAMS = dict(objective='lambdarank', metric='None', learning_rate=.05,
    num_leaves=63, min_data_in_leaf=200, feature_fraction=.9,
    bagging_fraction=.9, bagging_freq=1, label_gain=[0,1,3,7],
    verbosity=-1, seed=SEED, num_threads=THREADS, deterministic=True, force_col_wise=True)

def fit_ranker(feat, columns):
    tr = feat[feat.split=='train'].sort_values(['session_id','cand_wid']).reset_index(drop=True)
    va = feat[feat.split=='val'].sort_values(['session_id','cand_wid']).reset_index(drop=True)
    groups = lambda d:d.groupby('session_id',sort=False).size().to_numpy()
    train = lgb.Dataset(tr[columns], label=tr.label, group=groups(tr))
    valid = lgb.Dataset(va[columns], label=va.label, group=groups(va), reference=train)
    return lgb.train(LGB_PARAMS,train,num_boost_round=600,valid_sets=[valid],
        feval=lightgbm_metric(va),
        callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(100)])

def fold_in_svd(hist, train_ids, n_sessions, dimensions=64):
    """Fit shared item factors on training histories, transform other histories only."""
    train_hist = hist[hist.session_id.isin(train_ids)]
    items = {w:i for i,w in enumerate(sorted(train_hist.wid.unique()))}
    mapped = hist.wid.map(items); keep = mapped.notna()
    matrix = csr_matrix((np.ones(int(keep.sum()),dtype=np.float32),
        (hist.loc[keep,'session_id'].to_numpy(),mapped[keep].astype(int).to_numpy())),
        shape=(n_sessions,len(items)),dtype=np.float32)
    k = min(dimensions,len(train_ids)-1,len(items)-1)
    if k < 1:
        raise ValueError('Insufficient training users/items for SVD')
    svd = TruncatedSVD(n_components=k,random_state=SEED)
    svd.fit(matrix[sorted(train_ids)])
    return items,svd,svd.transform(matrix)

def main():
    prepare(); start=time.time()
    feat=pd.read_parquet(OUT/'features.parquet')
    hist=pd.read_parquet(OUT/'history.parquet')
    sess=pd.read_parquet(OUT/'sessions.parquet')
    test=feat[feat.split=='test'].copy()
    results={}; detail={}
    def report(name,df,col):
        m,n=evaluate(df,col); results[name]=m
        detail[name]={'queries_with_positive':n,'all_queries':int(df.session_id.nunique())}
        print(name,m,flush=True)
    test['random_score']=tie_keys(test,SEED+1).astype(float)/np.iinfo(np.uint64).max
    test['content_score']=test.ov_name+test.ov_cat+.5*test.ov_brand+test.jacc_name
    report('Random',test,'random_score')
    report('Popularity',test,'pop_wid')
    report('Content/Relevance-only',test,'content_score')
    train_ids=set(sess.loc[sess.split=='train','session_id'])
    items,svd,user_factors=fold_in_svd(hist,train_ids,int(sess.session_id.max())+1)
    factors=svd.components_.T
    mapped=test.cand_wid.map(items)
    scores=np.zeros(len(test),dtype=np.float32); warm=mapped.notna().to_numpy()
    ids=mapped.fillna(0).astype(int).to_numpy()
    # Bounded temporary arrays avoid duplicating a whole candidate x factor matrix.
    sids=test.session_id.to_numpy()
    for a in range(0,len(test),20000):
        b=min(a+20000,len(test)); ok=warm[a:b]
        block=np.zeros(b-a,dtype=np.float32)
        block[ok]=np.einsum('ij,ij->i',user_factors[sids[a:b][ok]],factors[ids[a:b][ok]])
        scores[a:b]=block
    test['svd_score']=scores
    report('SVD(train-only fold-in)',test,'svd_score')
    joblib.dump({'svd':svd,'item_ids':items},OUT/'svd.joblib',compress=3)
    columns=feature_columns(feat)
    booster=fit_ranker(feat,columns)
    test['lgb_score']=booster.predict(test[columns])
    report('LambdaMART(features)',test,'lgb_score')
    for split in ['train','val']:
        frame=feat[feat.split==split].copy()
        frame['lgb_score']=booster.predict(frame[columns])
        report(f'LambdaMART(features)_{split}',frame,'lgb_score')
        report(f'Popularity_{split}',frame,'pop_wid')
    pd.DataFrame({'feature':columns,'gain':booster.feature_importance('gain')}).sort_values(
        'gain',ascending=False).to_csv(RES/'lgb_feature_importance.csv',index=False)
    booster.save_model(str(OUT/'lgb_model.txt'))
    test[['session_id','cand_wid','label','lgb_score','svd_score','random_score',
          'content_score','pop_wid']].to_parquet(OUT/'test_pred_lgb.parquet',index=False)
    save_json(OUT/'lgb_preprocessing.json',{'feature_columns':columns,'schema_version':2})
    save_json(RES/'results_baselines_lgb.json',results)
    save_json(RES/'baseline_protocol.json',dict(query_counts=detail,seed=SEED,
        svd_train_history_item_coverage=float(warm.mean()),lgb_parameters=LGB_PARAMS,
        best_iteration=booster.best_iteration,
        evaluation='macro full-slate metrics, positive queries only, deterministic ID tie keys',
        svd='train-history item factors; held-out preceding history folded into fixed factors'))
    print(f'Step 4 complete ({time.time()-start:.0f}s)',flush=True)

if __name__=='__main__':
    main()
```


In [ ]:
run_stage('4')


## 7 Neural models and matched tower controls

Read fit_preprocessing and encode_frames, then pack, DHAR, HistoryCF,
listwise and train_one in that order. Training-only vocabularies and
normalization are saved. The brand hash is deterministic.

Numeric input is [batch, slate, features]. Candidate embeddings are
[batch, slate, 32]. Each history channel holds up to 20 sampled known-category
positions. Candidate-category attention supplies short/order context vectors.

The MLP, pooled, short-only, order-only and dual variants share numeric
features and a matched training procedure. The MLP retains engineered history
features, isolating the incremental sequence towers. Training slates use up
to 64 candidates with all positives retained; evaluation uses complete slates.

HistoryCF is a GMF/MLP adaptation with a pooled preceding-item encoder,
not a faithful user-ID NCF reproduction. It can represent an unseen user
without an untrained session-ID embedding.


### Source to read: step5_neural.py

[Canonical source at the pinned revision](https://github.com/steffanylars/nus-web-mining-business-insights/blob/4739ff8fe2f59a4141310d60fc36bd207517b0c4/Final%20Project/src/step5_neural.py). The following is a readable source snapshot, not an executed training cell. The run cell below calls this script from the checked-out repository.

```python
"""History-conditioned neural ranking and matched history-tower ablations.

All shared parameters/preprocessing use training users only. Held-out users are
represented by their preceding history, never by an untrained user-ID embedding.
"""
import argparse
import copy
import hashlib
import json
import math
import random
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from config import (OUT, RES, SEED, HISTORY_LENGTH, EPOCHS, BATCH_SIZE, THREADS,
                    feature_columns, prepare, save_json)
from data_utils import short_mask, select_sequence
from eval_utils import evaluate

VARIANTS = ['mlp','pooled','short_only','order_only','dual','history_cf']
NAMES = {'mlp':'MLP(no towers)','pooled':'Pooled history','short_only':'Short only',
         'order_only':'Order only','dual':'DHAR(ours)','history_cf':'History-conditioned CF'}
BRAND_BUCKETS = 20000

def device():
    return torch.device('cuda' if torch.cuda.is_available() else
                        'mps' if torch.backends.mps.is_available() else 'cpu')

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def brand_bucket(value):
    if value is None or pd.isna(value) or str(value).strip() in {'','-1'}:
        return 0
    digest=hashlib.blake2b(str(value).encode(),digest_size=8).digest()
    return int.from_bytes(digest,'little')%(BRAND_BUCKETS-1)+1

def fit_preprocessing(feat,hist,meta,length):
    cols=feature_columns(feat); train=feat.split.eq('train')
    train_ids=set(feat.loc[train,'session_id'])
    train_hist=hist[hist.session_id.isin(train_ids)]
    train_wids=set(feat.loc[train,'cand_wid'])|set(train_hist.wid)
    tm=meta[meta.wid.isin(train_wids)]
    categories=sorted(c for c in tm.cate_id_3.dropna().unique() if str(c).strip())
    mu=feat.loc[train,cols].mean()
    sd=feat.loc[train,cols].std().fillna(1).replace(0,1)
    return dict(schema_version=2,feature_columns=cols,mean=mu.tolist(),std=sd.tolist(),
        category_ids={str(c):i+1 for i,c in enumerate(categories)},
        item_ids={w:i+1 for i,w in enumerate(sorted(train_wids))},
        history_length=length,cf_history_length=2*length,brand_buckets=BRAND_BUCKETS,
        brand_hash='blake2b UTF-8, 8 bytes little-endian; modulo 19999 + 1',
        sequence_policy='evenly spaced positions over each complete chronological horizon',
        short_policy='CLICK <=7d or CART <=90d',order_policy='all preceding ORD events')

def encode_frames(feat,hist,meta,prep):
    frame=feat.copy()
    cols=prep['feature_columns']
    frame[cols]=((frame[cols]-np.array(prep['mean']))/np.array(prep['std'])).astype('float32')
    if not np.isfinite(frame[cols].to_numpy()).all():
        raise ValueError('Nonfinite neural features')
    maps=meta.set_index('wid')
    cat=maps.cate_id_3.to_dict(); brand=maps.brand_id.to_dict()
    frame['_c3']=frame.cand_wid.map(cat).map(prep['category_ids']).fillna(0).astype(int)
    frame['_bi']=frame.cand_wid.map(brand).map(brand_bucket).astype(int)
    frame['_ii']=frame.cand_wid.map(prep['item_ids']).fillna(0).astype(int)
    h=hist.sort_values(['session_id','pos']).copy()
    h['_c3']=h.wid.map(cat).map(prep['category_ids']).fillna(0).astype(int)
    h['_ii']=h.wid.map(prep['item_ids']).fillna(0).astype(int)
    sequences={}; coverage={}
    for name,subset,col,length in [
        ('short',h[short_mask(h)],'_c3',prep['history_length']),
        ('order',h[h.type.eq('ORD')],'_c3',prep['history_length']),
        ('cf',h,'_ii',prep['cf_history_length'])]:
        seq={}; eligible=retained=unknown=0
        for sid,g in subset.groupby('session_id',sort=False):
            values=g[col].to_numpy()
            eligible+=len(values); unknown+=int((values==0).sum())
            # Unknown metadata/items are excluded before position sampling.
            selected=select_sequence(values[values>0],length)
            seq[sid]=selected; retained+=len(selected)
        sequences[name]=seq
        coverage[name]={'eligible_events':eligible,'unknown_events':unknown,'selected_events':retained,
                        'users_with_known_sequence':sum(bool(len(v)) for v in seq.values())}
    return frame,sequences,coverage

def pack(frame,sequences,prep,split,width=200,seed=SEED):
    cols=prep['feature_columns']; groups=frame[frame.split.eq(split)].groupby('session_id',sort=True)
    n=len(groups); nf=len(cols); L=prep['history_length']; H=prep['cf_history_length']
    arrays=dict(num=np.zeros((n,width,nf),dtype=np.float32),
        c3=np.zeros((n,width),dtype=np.int64),bi=np.zeros((n,width),dtype=np.int64),
        ii=np.zeros((n,width),dtype=np.int64),lab=np.zeros((n,width),dtype=np.int64),
        mask=np.zeros((n,width),dtype=bool),ridx=np.full((n,width),-1,dtype=np.int64),
        short=np.zeros((n,L),dtype=np.int64),order=np.zeros((n,L),dtype=np.int64),
        cf=np.zeros((n,H),dtype=np.int64))
    for row,(sid,g) in enumerate(groups):
        # Canonical order prevents source positions influencing sampling/roundoff.
        g=g.sort_values('cand_wid')
        if len(g)>width:
            if split!='train':
                raise ValueError('Evaluation must use every supplied candidate')
            rng=np.random.default_rng(np.random.SeedSequence([seed,int(sid)]))
            pos=np.flatnonzero(g.label.to_numpy()>0); neg=np.flatnonzero(g.label.to_numpy()==0)
            if len(pos)>width:
                raise ValueError('Training width is smaller than number of positives')
            selected=np.r_[pos,rng.choice(neg,width-len(pos),replace=False)]
            g=g.iloc[selected].sort_values('cand_wid')
        m=len(g)
        arrays['num'][row,:m]=g[cols].to_numpy(dtype=np.float32)
        for key,col in [('c3','_c3'),('bi','_bi'),('ii','_ii'),('lab','label')]:
            arrays[key][row,:m]=g[col].to_numpy()
        arrays['mask'][row,:m]=True; arrays['ridx'][row,:m]=g.index.to_numpy()
        for key in ['short','order','cf']:
            v=sequences[key].get(sid,[])
            arrays[key][row,:len(v)]=v
    return {k:torch.from_numpy(v) for k,v in arrays.items()}

class DHAR(nn.Module):
    def __init__(self,nc3,nb,nf,variant='dual',d=32,**unused):
        super().__init__(); self.variant=variant
        self.ec3=nn.Embedding(nc3,d,padding_idx=0); self.eb=nn.Embedding(nb,d,padding_idx=0)
        self.ps=nn.Linear(d,d); self.pl=nn.Linear(d,d)
        self.head=nn.Sequential(nn.Linear(4*d+nf,128),nn.ReLU(),nn.Dropout(.2),
            nn.Linear(128,64),nn.ReLU(),nn.Dropout(.2),nn.Linear(64,1))
    def attend(self,q,seq,projection):
        emb=self.ec3(seq); mask=seq>0
        score=torch.einsum('bsd,bld->bsl',projection(q),emb)/math.sqrt(emb.size(-1))
        score=score.masked_fill(~mask[:,None,:],-1e9)
        weight=torch.softmax(score,dim=-1)*mask[:,None,:]
        weight=weight/weight.sum(-1,keepdim=True).clamp_min(1e-9)
        return torch.einsum('bsl,bld->bsd',weight,emb)
    def forward(self,num,c3,bi,ii,short,order,cf):
        qc=self.ec3(c3); qb=self.eb(bi); sc=torch.zeros_like(qc); lc=torch.zeros_like(qc)
        if self.variant in {'dual','short_only'}:
            sc=self.attend(qc,short,self.ps)
        if self.variant in {'dual','order_only'}:
            lc=self.attend(qc,order,self.pl)
        if self.variant=='pooled':
            both=torch.cat([short,order],dim=1); mask=both>0
            pool=(self.ec3(both)*mask[:,:,None]).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)
            sc=pool[:,None,:].expand_as(qc); lc=sc
        return self.head(torch.cat([qc,qb,sc,lc,num],dim=-1)).squeeze(-1)

class HistoryCF(nn.Module):
    """GMF+MLP with a mean history encoder instead of per-session embeddings."""
    def __init__(self,ni,d=32,**unused):
        super().__init__()
        self.item=nn.Embedding(ni,d,padding_idx=0)
        self.mlp=nn.Sequential(nn.Linear(2*d,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU())
        self.out=nn.Linear(d+32,1)
    def forward(self,num,c3,bi,ii,short,order,cf):
        mask=cf>0
        user=(self.item(cf)*mask[:,:,None]).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)
        candidate=self.item(ii); user=user[:,None,:].expand_as(candidate)
        return self.out(torch.cat([user*candidate,self.mlp(torch.cat([user,candidate],-1))],-1)).squeeze(-1)

def listwise(scores,lab,mask):
    gain=torch.tensor([0.,1.,3.,7.],device=scores.device)[lab]*mask
    target=gain/gain.sum(1,keepdim=True).clamp_min(1e-9)
    loss=-(target*F.log_softmax(scores.masked_fill(~mask,-1e9),dim=1)).sum(1)
    positive=gain.sum(1)>0
    return loss[positive].mean() if positive.any() else scores.sum()*0

def forward_batch(model,pk,indices,dev):
    return model(*[pk[k][indices].to(dev) for k in ['num','c3','bi','ii','short','order','cf']])

def predict_pack(model,pk,n_rows,dev):
    model.eval(); scores=np.zeros(n_rows,dtype=np.float32)
    with torch.no_grad():
        for a in range(0,len(pk['num']),BATCH_SIZE):
            sl=slice(a,a+BATCH_SIZE)
            value=forward_batch(model,pk,sl,dev).cpu().numpy()
            ids=pk['ridx'][sl].numpy(); mask=pk['mask'][sl].numpy()
            scores[ids[mask]]=value[mask]
    return scores

def train_one(variant,prep,frame,packs,seed,dev):
    seed_everything(seed)
    spec=dict(nc3=len(prep['category_ids'])+1,nb=prep['brand_buckets'],
              nf=len(prep['feature_columns']),ni=len(prep['item_ids'])+1,variant=variant)
    model=(HistoryCF(**spec) if variant=='history_cf' else DHAR(**spec)).to(dev)
    optimizer=torch.optim.Adam(model.parameters(),lr=.002,weight_decay=1e-6)
    best=-1.; state=None; bad=0; log=[]; start=time.time()
    val=frame[frame.split.eq('val')].copy()
    tr=packs['train']
    for epoch in range(1,EPOCHS+1):
        model.train(); perm=np.random.permutation(len(tr['num'])); total=0.; batches=0
        for a in range(0,len(perm),BATCH_SIZE):
            ix=perm[a:a+BATCH_SIZE]
            score=forward_batch(model,tr,ix,dev)
            loss=listwise(score,tr['lab'][ix].to(dev),tr['mask'][ix].to(dev))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total+=float(loss.detach()); batches+=1
        scores=predict_pack(model,packs['val'],len(frame),dev)
        val['score']=scores[val.index.to_numpy()]
        metrics,n=evaluate(val,'score'); ndcg=metrics['NDCG@10']
        log.append(dict(epoch=epoch,loss=total/max(batches,1),val_ndcg10=ndcg))
        print(f'{variant} seed={seed} epoch={epoch} val={ndcg:.5f} elapsed={time.time()-start:.0f}s',flush=True)
        if ndcg>best:
            best=ndcg; state=copy.deepcopy({k:v.detach().cpu() for k,v in model.state_dict().items()}); bad=0
        else:
            bad+=1
            if bad>=3: break
    model.load_state_dict(state)
    name=f'{variant}_seed{seed}'
    torch.save({'state_dict':state,'spec':spec,'schema_version':2},OUT/f'{name}.pt')
    save_json(RES/f'training_{name}.json',dict(seed=seed,variant=variant,epochs=log,
        best_validation_ndcg10=best,training_slate_width=packs['train']['mask'].shape[1],
        evaluation_slate_width=200,batch_size=BATCH_SIZE,device=str(dev)))
    return model

def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--variants',nargs='+',choices=VARIANTS,default=VARIANTS)
    parser.add_argument('--seeds',nargs='+',type=int,default=[SEED])
    args=parser.parse_args()
    prepare(); torch.set_num_threads(THREADS); dev=device()
    print('device',dev,flush=True)
    feat=pd.read_parquet(OUT/'features.parquet')
    hist=pd.read_parquet(OUT/'history.parquet')
    meta=pd.read_parquet(OUT/'meta_subset.parquet')
    prep=fit_preprocessing(feat,hist,meta,HISTORY_LENGTH)
    save_json(OUT/'neural_preprocessing.json',prep)
    frame,sequences,coverage=encode_frames(feat,hist,meta,prep)
    save_json(RES/'horizon_coverage.json',coverage)
    # Same fixed negative sample and complete eval slates for every architecture.
    packs={s:pack(frame,sequences,prep,s,64 if s=='train' else 200)
           for s in ['train','val','test']}
    results={}; predictions=feat.loc[feat.split.eq('test'),['session_id','cand_wid','label']].copy()
    for seed in args.seeds:
        for variant in args.variants:
            model=train_one(variant,prep,frame,packs,seed,dev)
            key=NAMES[variant] if len(args.seeds)==1 else f'{NAMES[variant]} seed{seed}'
            test_scores=predict_pack(model,packs['test'],len(frame),dev)
            col=f'{variant}_score' if len(args.seeds)==1 else f'{variant}_seed{seed}_score'
            predictions[col]=test_scores[predictions.index.to_numpy()]
            m,n=evaluate(predictions,col); results[key]=m
            print('TEST',key,m,flush=True)
            val=frame[frame.split.eq('val')].copy()
            score=predict_pack(model,packs['val'],len(frame),dev)
            val['score']=score[val.index.to_numpy()]
            results[key+'_val']=evaluate(val,'score')[0]
            # Incremental saves allow a long CPU run to be resumed/reviewed.
            predictions.to_parquet(OUT/'test_pred_neural.parquet',index=False)
            save_json(RES/'results_neural.json',results)
            del model
    print('Neural variants complete',flush=True)

if __name__=='__main__':
    main()
```


In [ ]:
run_stage('5')


## 8 Diagnostics and interpretation

Rank each full slate before measuring item subgroups. Removing all warm
products before assessing cold products would change their competitors.
Distinguish absence from training history from absence from training
candidate rows. Limited-history users are not necessarily new customers.

Compare the dual model with the same-feature MLP and pooled-history
control. The separate LambdaMART ablation tests a temporal/action feature
bundle, not the neural towers. Bootstrap intervals are conditional on
saved models and do not estimate training-seed variation.


### Source to read: step6_diagnostics.py

[Canonical source at the pinned revision](https://github.com/steffanylars/nus-web-mining-business-insights/blob/4739ff8fe2f59a4141310d60fc36bd207517b0c4/Final%20Project/src/step6_diagnostics.py). The following is a readable source snapshot, not an executed training cell. The run cell below calls this script from the checked-out repository.

```python
"""Full-slate subgroup ranks, paired uncertainty and matched feature ablation."""
import argparse
import numpy as np
import pandas as pd
from config import OUT, RES, SEED, feature_columns, prepare, save_json
from eval_utils import evaluate, ranked_order, per_query
from step4_baselines_lgbm import fit_ranker, LGB_PARAMS

TEMPORAL_FEATURES=['aff_cat_decay','aff_brand_decay','short_cat_hit','short_brand_hit',
    'long_cat_hit','long_brand_hit','h_min_elapsed','h_n_recent7','h_n_ord','h_n_cart','h_n_click']

def positive_rank_records(pred,score,history_seen,training_candidates):
    records=[]
    for sid,g in pred.groupby('session_id',sort=True):
        order=ranked_order(g,score); ranks=np.empty(len(g),dtype=int)
        ranks[order]=np.arange(1,len(g)+1)
        for w,lab,rk in zip(g.cand_wid,g.label,ranks):
            if lab>0:
                records.append(dict(session_id=int(sid),cand_wid=w,label=int(lab),rank=int(rk),
                    training_history_seen=w in history_seen,
                    training_candidate_seen=w in training_candidates))
    return pd.DataFrame(records)

def bootstrap_difference(first,second,score_first,score_second,repetitions=2000):
    a=per_query(first,score_first).set_index('session_id')['NDCG@10']
    b=per_query(second,score_second).set_index('session_id')['NDCG@10']
    if set(a.index)!=set(b.index):
        raise ValueError('Paired comparison requires identical evaluable queries')
    delta=b.sort_index().to_numpy()-a.sort_index().to_numpy()
    rng=np.random.default_rng(SEED)
    boot=[rng.choice(delta,len(delta),replace=True).mean() for _ in range(repetitions)]
    return dict(delta=float(delta.mean()),ci95=np.quantile(boot,[.025,.975]).tolist(),
        positive_queries=len(delta),resamples=repetitions,seed=SEED,
        interpretation='conditional on fixed saved models; not training-seed uncertainty')

def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--skip-feature-ablation',action='store_true')
    args=parser.parse_args()
    prepare()
    feat=pd.read_parquet(OUT/'features.parquet')
    sess=pd.read_parquet(OUT/'sessions.parquet')
    hist=pd.read_parquet(OUT/'history.parquet')
    lp=pd.read_parquet(OUT/'test_pred_lgb.parquet')
    npred=pd.read_parquet(OUT/'test_pred_neural.parquet')
    keys=['session_id','cand_wid']
    merged=lp[keys+['label']].merge(npred[keys+['label']],on=keys,validate='one_to_one',suffixes=('_a','_b'))
    if len(merged)!=len(lp) or len(merged)!=len(npred) or not (merged.label_a==merged.label_b).all():
        raise ValueError('Prediction populations differ')
    train_ids=set(sess.loc[sess.split.eq('train'),'session_id'])
    history_seen=set(hist.loc[hist.session_id.isin(train_ids),'wid'])
    candidate_seen=set(feat.loc[feat.split.eq('train'),'cand_wid'])
    models=[('LambdaMART',lp,'lgb_score')]
    models.extend((c.removesuffix('_score'),npred,c) for c in npred if c.endswith('_score'))
    segments=[]; rank_tables=[]; recalls=[]
    history_bins=pd.cut(sess.n_hist,[-1,2,20,100,np.inf],
        labels=['limited(0-2)','short(3-20)','medium(21-100)','long(>100)'])
    history_map=dict(zip(sess.session_id,history_bins.astype(str)))
    for name,pred,col in models:
        frame=pred.copy(); frame['history_group']=frame.session_id.map(history_map)
        for group,g in frame.groupby('history_group',sort=True):
            m,n=evaluate(g,col)
            segments.append(dict(model=name,history_group=group,positive_queries=n,
                all_queries=int(g.session_id.nunique()),**m))
        rr=positive_rank_records(pred,col,history_seen,candidate_seen)
        rr['model']=name; rank_tables.append(rr)
        for dimension in ['training_history_seen','training_candidate_seen']:
            for seen,g in rr.groupby(dimension):
                by_query=g.groupby('session_id')['rank'].apply(lambda x:float((x<=10).mean()))
                recalls.append(dict(model=name,dimension=dimension,seen=bool(seen),
                    positive_items=len(g),queries_with_subgroup_positive=len(by_query),
                    macro_recall10=float(by_query.mean()),micro_recall10=float(g['rank'].le(10).mean()),
                    mean_full_slate_rank=float(g['rank'].mean()),median_full_slate_rank=float(g['rank'].median())))
    pd.DataFrame(segments).to_csv(RES/'segment_metrics.csv',index=False)
    pd.DataFrame(recalls).to_csv(RES/'positive_subgroup_full_slate.csv',index=False)
    allr=pd.concat(rank_tables,ignore_index=True)
    allr.groupby(['model','label'])['rank'].agg(['mean','median','count']).to_csv(RES/'rank_by_type.csv')
    if 'dual_score' in npred:
        save_json(RES/'paired_bootstrap.json',dict(
            dual_minus_lgb=bootstrap_difference(lp,npred,'lgb_score','dual_score'),
            dual_minus_mlp=bootstrap_difference(npred,npred,'mlp_score','dual_score'),
            dual_minus_pooled=bootstrap_difference(npred,npred,'pooled_score','dual_score')))
    if not args.skip_feature_ablation:
        # Full arm is the headline model, not an unrelated retraining configuration.
        cols=feature_columns(feat)
        booster=fit_ranker(feat,[c for c in cols if c not in TEMPORAL_FEATURES])
        test=feat[feat.split.eq('test')].copy()
        test['score']=booster.predict(test[[c for c in cols if c not in TEMPORAL_FEATURES]])
        full=evaluate(lp,'lgb_score')[0]; without=evaluate(test,'score')[0]
        save_json(RES/'ablation.json',dict(full=full,no_temporal=without,
            delta_NDCG10=full['NDCG@10']-without['NDCG@10'],removed_features=TEMPORAL_FEATURES,
            params=LGB_PARAMS,interpretation='temporal/action feature bundle, not a DHAR tower ablation'))
    save_json(RES/'diagnostic_protocol.json',dict(
        subgroup_ranks='rank every candidate first; measure positive subgroup ranks/recall in original full slate',
        history_seen='present in complete training histories',
        candidate_seen='present in training candidate rows, regardless of label',
        low_history='limited history, not proof of a new user',
        no_zero_history_claim='no natural zero-history benchmark is claimed'))
    print('Full-slate diagnostics and feature ablation complete',flush=True)

if __name__=='__main__':
    main()
```


In [ ]:
run_stage('6', 'report')


## 9 Checks and discussion

The regression suite targets the original failure modes. The optional
completed-run verifier additionally requires a full 20,000-user run and
local checkpoints; it should not be applied to a 1,000-user trial.

Questions to discuss:

- Why is a user split appropriate, and how does it differ from the paper's protocol?
- Which inputs are available before current outcomes, and what is fit only on training data?
- Why do complete histories matter even though neural sequences remain bounded?
- What does the MLP comparison tell us about added architecture complexity?
- Which failure cases and resource measurements should the final report include?

The inspected test set is not a new untouched holdout. Offline ranking does
not establish conversion/revenue lift or causal shopper behavior.


In [ ]:
if RUN_REGRESSION_CHECKS:
    subprocess.run([EXPERIMENT_PYTHON, '-m', 'unittest', 'discover',
                    '-s', str(PROJECT / 'tests'), '-v'],
                   cwd=REPO_DIR, env=experiment_env, check=True)
else:
    print('Regression tests not rerun; enable RUN_REGRESSION_CHECKS after installing dependencies.')

if RUN_EXPERIMENTS:
    display(Markdown((RUN_DIR / 'RESULTS.md').read_text(encoding='utf-8')))
else:
    print('No training was requested; the tables above remain the historical saved results.')


## Team reference links

- [Change log](https://github.com/steffanylars/nus-web-mining-business-insights/blob/ethan/Final%20Project/ETHAN_CHANGES.md)
- [Corrected report](https://github.com/steffanylars/nus-web-mining-business-insights/blob/ethan/Final%20Project/artifacts/corrected/REPORT.md)
- [A+ reference improvements](https://github.com/steffanylars/nus-web-mining-business-insights/blob/ethan/Final%20Project/APLUS_REFERENCE_IMPROVEMENTS.md)
- [Reproduction README](https://github.com/steffanylars/nus-web-mining-business-insights/blob/ethan/Final%20Project/README.md)

This notebook is a review and execution interface for the existing pipeline,
not an additional model experiment. Source snapshots correspond to commit
`4739ff8fe2f59a4141310d60fc36bd207517b0c4`; regenerate the notebook if the canonical source changes.
